# Install Depndancy

In [ ]:
! pip install -q json-repair  qwen-vl-utils python-docx bitsandbytes hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 75.3 MB/s eta 0:00:00


In [ ]:
!pip install vllm==0.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 123.9 M

# Snapshot Download

# VLM

## Load Data

In [ ]:
import os
DRIVE_BASE    = '/content/drive/MyDrive/Iran Israel War/extracted_map_layers'    # your project folder
# INPUT_CSV     = os.path.join(DRIVE_BASE, 'final_extracted_events.csv')
INPUT_CSV     = os.path.join(DRIVE_BASE, 'master_unified_campaign_log.csv')
OUTPUT_CSV    = os.path.join(DRIVE_BASE, 'bda_assessed_final_report.csv')
OUTPUT_DOCX   = os.path.join(DRIVE_BASE, 'BDA_Final_Dossier.docx')


os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {VLM_MODEL_ID}')

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War/extracted_map_layers


In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)

In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)
df_raw.rename(columns={'final_calculated_radius_m':'max_crater_radius_m',"latitude":'lat_dec','longitude':'lon_dec'},inplace=True)
# Keep only rows that have both image paths
valid_analysis_df = df_raw.copy().reset_index(drop=True)

# print(f'✅ Total rows     : {len(df_raw)}')
# print(f'   Processable   : {len(valid_analysis_df)}')
# print(f'   Skipped       : {len(df_raw) - len(valid_analysis_df)} (missing image paths)')
# valid_analysis_df[['event_full_title', 'image_path_sat', 'image_path_map']].head()

In [ ]:
valid_analysis_df.columns

Index(['campaign_phase', 'original_source_row', 'event_id', 'strikedate',
       'actor', 'side', 'event_type', 'site_type', 'siteStype', 'country',
       'lat_dec', 'lon_dec', 'max_crater_radius_m', 'impact_footprint_sqm'],
      dtype='object')

In [ ]:
df_raw.index

RangeIndex(start=0, stop=893, step=1)

## Load VLM

In [ ]:
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
from docx import Document
from docx.shared import Inches
from transformers import AutoProcessor
import torch
# ==============================================================================
# PHASE 2: RIGID VLM ASSESSMENT (TWO IMAGES)
# ==============================================================================
print("\n--- PHASE 2: NATIVE MULTIMODAL BDA ---")

num_gpus = torch.cuda.device_count()
print(f"[INFO] Detected {num_gpus} GPUs. Splitting model across them...")




--- PHASE 2: NATIVE MULTIMODAL BDA ---
[INFO] Detected 1 GPUs. Splitting model across them...


## Prompt

In [ ]:
# ============================================================
# VLM PROMPTS — TWO VERSIONS
#
# VERSION A — COMPACT (no reasoning)
#   Use for: main pipeline, all 5 runs
#   max_tokens: 800
#   Best when: you trust the median to catch errors
#
# VERSION B — REASONING (one reasoning line per building)
#   Use for: flagged rows (high std dev), audit runs, QA
#   max_tokens: 3000
#   Best when: you need to understand WHY a building was counted
#
# Suggested workflow:
#   1. Run VERSION A × 5, take median → fast bulk pass
#   2. Flag rows where std dev > 1.5
#   3. Run VERSION B × 1 on flagged rows → human readable audit
# ============================================================

# ── VERSION A — COMPACT ──────────────────────────────────────

vlm_prompt_compact = """\
You are a Geospatial Intelligence Analyst. Two satellite images:
  1. Google Satellite (oblique) — 3D confirmation
  2. ESRI Satellite (nadir)    — primary for all boundary decisions

List every intact building roof you can see, one line each.
Skip only: trees, cars, shadows, ruins. Split only if a visible gap or parapet separates them.

Under "Buildings:" write one line per building:
B<n> <clock-face> <Quadrant>: <3-word roof description> | <Fully Inside | Partially Inside | Outside>

Verdict definitions:
  Fully Inside    : entire roof under red shading (touching boundary = Inside)
  Partially Inside: red boundary visibly cuts through the roof
  Outside         : roof entirely on unshaded ground

If no buildings are visible at all, write "Empty" under Buildings.

Then immediately output the block:
FULLY_INSIDE: {integer}
PARTIALLY_INSIDE: {integer}
TOTAL_IMPACTED: {integer}
MAP_TEXT: {labels inside red zone verbatim, or None}
==================END==================

EXAMPLE 1 — mixed verdicts:
Buildings:
B1 2-o'clock Top-Right: flat concrete roof | Fully Inside
B2 5-o'clock Bottom-Right: small metal annex | Partially Inside
B3 9-o'clock Top-Left: wide warehouse roof | Fully Inside
B4 11-o'clock Top-Left: narrow brick structure | Outside
FULLY_INSIDE: 2
PARTIALLY_INSIDE: 1
TOTAL_IMPACTED: 3
MAP_TEXT: مدرسة / School
==================END==================

EXAMPLE 2 — nothing visible:
Buildings:
Empty
FULLY_INSIDE: 0
PARTIALLY_INSIDE: 0
TOTAL_IMPACTED: 0
MAP_TEXT: None
==================END==================
"""

KICKSTART_COMPACT = "<think>\nBuildings:\n"


# ── VERSION B — REASONING ────────────────────────────────────

vlm_prompt_reasoning = """\
You are a Geospatial Intelligence Analyst. Two satellite images:
  1. Google Satellite (oblique) — 3D confirmation
  2. ESRI Satellite (nadir)    — primary for all boundary decisions

List every intact building roof you can see.
Skip only: trees, cars, shadows, ruins. Split only if a visible gap or parapet separates them.

Under "Buildings:" write two lines per building:
B<n> <clock-face> <Quadrant>: <3-word roof description> | <Fully Inside | Partially Inside | Outside>
  WHY: <which image used> | <what confirms it is a building> | <how red zone interacts with roof>

Verdict definitions:
  Fully Inside    : entire roof under red shading (touching boundary = Inside)
  Partially Inside: red boundary visibly cuts through the roof
  Outside         : roof entirely on unshaded ground

If no buildings are visible at all, write "Empty" under Buildings.

Then immediately output the block:
FULLY_INSIDE: {integer}
PARTIALLY_INSIDE: {integer}
TOTAL_IMPACTED: {integer}
MAP_TEXT: {labels inside red zone verbatim, or None}
==================END==================

EXAMPLE 1 — mixed verdicts:
Buildings:
B1 2-o'clock Top-Right: flat concrete roof | Fully Inside
  WHY: ESRI clearer | straight edges, hard shadow confirms walls | entire roof under red shading
B2 5-o'clock Bottom-Right: small metal annex | Partially Inside
  WHY: Google confirms 3D | metallic roof, distinct from ground | boundary cuts NE corner
B3 9-o'clock Top-Left: wide warehouse roof | Fully Inside
  WHY: ESRI clearer | uniform grey texture, rectangular | fully within red overlay
B4 11-o'clock Top-Left: narrow brick structure | Outside
  WHY: ESRI clearer | straight parapet edges visible | no red shading contact
FULLY_INSIDE: 2
PARTIALLY_INSIDE: 1
TOTAL_IMPACTED: 3
MAP_TEXT: مدرسة / School
==================END==================

EXAMPLE 2 — nothing visible:
Buildings:
Empty
FULLY_INSIDE: 0
PARTIALLY_INSIDE: 0
TOTAL_IMPACTED: 0
MAP_TEXT: None
==================END==================
"""

KICKSTART_REASONING = "<think>\nBuildings:\n"


# ── SUMMARY ──────────────────────────────────────────────────

print("✅ Both prompt versions defined")
print(f"\n[A] COMPACT  — {len(vlm_prompt_compact):,} chars | max_tokens=800")
print(f"    Kickstart : {repr(KICKSTART_COMPACT)}")
print(f"\n[B] REASONING — {len(vlm_prompt_reasoning):,} chars | max_tokens=3000")
print(f"    Kickstart : {repr(KICKSTART_REASONING)}")

✅ Both prompt versions defined

[A] COMPACT  — 1,534 chars | max_tokens=800
    Kickstart : '<think>\nBuildings:\n'

[B] REASONING — 1,966 chars | max_tokens=3000
    Kickstart : '<think>\nBuildings:\n'


In [ ]:
vlm_prompt = """\
You are a Geospatial Intelligence Analyst. Two satellite images:
  1. Google Satellite (oblique) — 3D confirmation
  2. ESRI Satellite (nadir)    — primary for all boundary decisions

Count every intact building roof intersecting the red-shaded zone.
Skip trees, cars, shadows, ruins. Split only if a visible gap or parapet separates them.

Verdict definitions:
  Fully Inside    : entire roof under red shading (touching boundary = Inside)
  Partially Inside: red boundary visibly cuts through the roof
  Outside         : roof entirely on unshaded ground

After your analysis, output this block exactly:
FULLY_INSIDE: {integer}
PARTIALLY_INSIDE: {integer}
TOTAL_IMPACTED: {integer}
MAP_TEXT: {labels inside red zone verbatim, or None}
==================END==================
"""

KICKSTART = "<think>\n"

STOP = ["==================END=================="]

In [ ]:
# # ============================================================
# # VLM PROMPT — FINAL COMPACT
# # ============================================================
# vlm_prompt = vlm_prompt_compact
# KICKSTART = KICKSTART_COMPACT

# print("✅ Prompt and kickstart defined")
# print(f"   Prompt length : {len(vlm_prompt):,} chars")
# print(f"   Kickstart     : {repr(KICKSTART)}")
# print(f"   Recommended max_tokens: 800")


## Build and Run Inference

In [ ]:

import os
# ── SAMPLING CONFIG ───────────────────────────────────────────────────────────
N_SAMPLES   = 5      # >1 required for median consensus to be meaningful
TEMPERATURE = 0.3    # low but non-zero so samples can differ
MAX_TOKENS  = 8192  # free reasoning needs room
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {model_id}')
print(f'   N samples  : {N_SAMPLES}')
print(f'   Temperature: {TEMPERATURE}')
print(f'   Max tokens : {MAX_TOKENS}')

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War/extracted_map_layers
   N samples  : 5
   Temperature: 0.3
   Max tokens : 8192


In [ ]:
# import shutil
# if not os.path.exists( '/content/images'):
#   shutil.copytree( os.path.join(DRIVE_BASE, "impact_maps_final"), '/content/images')

In [ ]:
# import shutil
# if not os.path.exists( '/content/images_2'):
#   shutil.copytree( os.path.join(DRIVE_BASE, "impact_maps_final_no_label"), '/content/images_2')

In [ ]:
# import os

# DEST = "/content/images"
# os.makedirs(DEST, exist_ok=True)
# if not os.path.exists( '/content/images'):
#   # Extract both zips
#   !unzip -q "impact_maps_final_no_label-20260611T123211Z-3-001" -d /content/temp_extract
#   # !unzip -q "/content/drive/MyDrive/crater_visualisations_part_2.zip" -d /content/temp_extract

#   # Move files from crater_visualisations into images
#   !mv /content/temp_extract/impact_maps_final_no_label/* "$DEST"/

#   # Remove temporary folder
#   !rm -rf /content/temp_extract

In [ ]:
import os

DEST = "/content/images"

if not os.path.exists(DEST):
    os.makedirs(DEST)

    # Extract both zips
    !unzip -q "/content/drive/MyDrive/Iran Israel War/extracted_map_layers/impact_maps_final_no_label-20260611T123211Z-3-001.zip" -d /content/temp_extract
  # !unzip -q "/content/drive/MyDrive/crater_visualisations_part_2.zip" -d /content/temp_extract


    # Move files into images
    !mv /content/temp_extract/impact_maps_final_no_label/* /content/images/

    # Cleanup
    !rm -rf /content/temp_extract

In [ ]:
# !ls /content/images/

In [ ]:
# ! rm -r /content/images

In [ ]:
# valid_analysis_df=valid_analysis_df.iloc[1:]

In [ ]:
from huggingface_hub import login
login("YOUR_HF_TOKEN")

In [ ]:
vlm_prompt = """\
You are a Geospatial Intelligence Analyst performing a Building Damage Assessment (BDA).
You are provided with a side-by-side satellite comparison:
  - Left Panel: Google Satellite (Source A)
  - Right Panel: ESRI Satellite (Source B)

TASK:
Count every distinct, intact building roof that intersects the area defined by the RED CIRCULAR BOUNDARY.
Use both images to cross-verify the existence of structures and resolve shadows or occlusions.

COUNTING RULES:
1. Count a building if any part of its roof touches or is inside the red ring.
2. Skip non-building objects: trees, vehicles, shadows, and debris/ruins.
3. Treat attached structures as a single building unless a visible gap or parapet wall clearly separates them.

VERDICT DEFINITIONS:
  - FULLY_INSIDE: The entire roof structure is within the red boundary.
  - PARTIALLY_INSIDE: The red boundary line visibly passes through the roof structure.
  - TOTAL_IMPACTED: The sum of the above.

After your analysis, output this block exactly:
FULLY_INSIDE: {integer}
PARTIALLY_INSIDE: {integer}
TOTAL_IMPACTED: {integer}
==================END==================
"""

KICKSTART = "<think>\n"

# ── SAMPLING CONFIG ───────────────────────────────────────────────────────────
N_SAMPLES   = 5      # >1 required for median consensus to be meaningful
TEMPERATURE = 0.3    # low but non-zero so samples can differ
MAX_TOKENS  = 2000  # free reasoning needs room
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {model_id}')
print(f'   N samples  : {N_SAMPLES}')
print(f'   Temperature: {TEMPERATURE}')
print(f'   Max tokens : {MAX_TOKENS}')

STOP = ["==================END=================="]
import os
import gc
import time
import torch
import shutil
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from huggingface_hub import snapshot_download

# IMPORT THE INTERNAL vLLM CLEANUP FUNCTION
from vllm.distributed.parallel_state import destroy_model_parallel

# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
# =========================
# HuggingFace cache (fast + persistent)
# =========================
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# =========================
# MinerU high-performance mode
# =========================
os.environ["MINERU_BACKEND"] = "vllm"
os.environ["MINERU_VLLM_ENABLE"] = "true"

# =========================
# vLLM stability + performance
# =========================
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
# os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "8192"

# =========================
# CUDA stability fixes
# =========================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

print("✅ Environment ready (vLLM performance mode)")

# ==============================================================================
# 2. DEFINE MODELS TO RUN
# ==============================================================================

SHORTER_LEN_MODELS=['deepseek-ai/deepseek-vl2','llava-hf/llava-v1.6-34b-hf',]

MODELS_TO_RUN = [
    #DONE
    #
    # 'google/gemma-4-31B-it',
    #  'zai-org/GLM-4.6V-Flash',
    # 'sakamakismile/Huihui-Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-NVFP4',
     "Qwen/Qwen3.6-35B-A3B",
    #  'nvidia/Cosmos-Reason2-32B',
    #######
    # 'mistralai/Mistral-Small-4-119B-2603-eagle',
    # 'meta-llama/Llama-4-Scout-17B-16E-Instruct',
    # 'DavidAU/Qwen3.6-40B-Claude-4.6-Opus-Deckard-Heretic-Uncensored-Thinking',

    # 'deepseek-ai/deepseek-vl2',



    # 'llava-hf/llava-v1.6-34b-hf',
    # 'zai-org/GLM-4.6V-FP8',

    # 'openai/gpt-oss-120b'
    # 'zai-org/GLM-4.6V-FP8'
    # 'meta-llama/Llama-3.2-90B-Vision',



]

DRIVE_BASE = "/content/drive/MyDrive/new/Outputs_2_no_labels" # Adjust to your actual path
os.makedirs(DRIVE_BASE, exist_ok=True)

# ==============================================================================
# 3. MAIN EXECUTION LOOP
# ==============================================================================
for model_id in MODELS_TO_RUN:
    print(f"\n{'='*60}")
    print(f"🚀 STARTING RUN FOR MODEL: {model_id}")
    print(f"{'='*60}")

    # --- Step A: Download Model ---
    print(f"\n[INFO] Downloading {model_id} to Disk...")
    snapshot_download(
        repo_id=model_id,
        resume_download=True,
        max_workers=8
    )
    print("[INFO] Download complete! Model is now cached on disk.")

    # --- Step B: Load Processor & VLM ---
    print(f'[INFO] Loading processor: {model_id}')
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    if model_id in SHORTER_LEN_MODELS:
      Max_len= 4096
    else:
      Max_len= 8192
    print(f'[INFO] Loading vLLM engine: {model_id}')
    vlm_llm = LLM(
        model=model_id,
        max_model_len=Max_len ,
        trust_remote_code=True,
        limit_mm_per_prompt={"image": 2},  # Allow 3 images per prompt
        gpu_memory_utilization=0.90,
        enforce_eager=True,
        disable_log_stats=False,
        # quantization="bitsandbytes",
        # load_format="bitsandbytes"
    )
    print('✅ VLM loaded successfully')

    # --- Step C: Build Prompt Inputs ---
    vlm_inputs = []
    missing_rows = []  # ✅ reset per model
    row_indices = []   # ✅ NEW: Track the indices of successful inputs

    try:
      valid_analysis_df.rename(columns={'latitude': 'lat_dec','longitude':'lon_dec'}, inplace=True)
    except:
      pass
    print(f'[INFO] Building inputs for {len(valid_analysis_df)} rows...')

    for idx, row in tqdm(valid_analysis_df.iterrows(), total=len(valid_analysis_df)):
        # Check image existence
        if not os.path.exists(os.path.join('/content/images', f"impact_{idx}_ari.jpg")):
          missing_rows.append(idx)
          continue
        if not os.path.exists(os.path.join('/content/images', f"impact_{idx}_sat.jpg")):
          missing_rows.append(idx)
          continue

        messages = [
            {
                'role': 'system',
                'content': (
                    'You are a strict, objective imagery analyst. '
                    'Only count clear, distinct, intact physical buildings. '
                    'Do not guess or infer structures that are not clearly visible.'
                ),
            },
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},  # Google (Ariel)
                    {'type': 'image'},  # Esri (Satellite)
                    {
                        'type': 'text',
                        'text': (
                            f"Location: {row['lat_dec']}, {row['lon_dec']}\n\n"
                            f"{vlm_prompt}"
                        ),
                    },
                ],
            },
        ]

        # ⚠️ REMOVED the broken idx=row[''] line from here

        # Load images (using the original DataFrame idx)
        img_ari = Image.open(os.path.join('/content/images', f"impact_{idx}_ari.jpg")).convert('RGB')
        img_sat = Image.open(os.path.join('/content/images', f"impact_{idx}_sat.jpg")).convert('RGB')

        # Apply template
        prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt += KICKSTART  # force the model straight into Step 1

        vlm_inputs.append({
            'prompt': prompt,
            'multi_modal_data': {'image': [img_ari, img_sat]},
        })
        row_indices.append(idx) # ✅ NEW: Save the index for this specific input

    print(f'✅ Built {len(vlm_inputs)} inputs')

    # --- Step D: Inference ---
    print(f'[INFO] Running VLM inference (n={N_SAMPLES}, temp={TEMPERATURE}, max_tokens={MAX_TOKENS})...')
    vlm_outputs = vlm_llm.generate(
        vlm_inputs,
        SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE, max_tokens=MAX_TOKENS),
        use_tqdm=True,
    )
    print(f'✅ Inference complete — {len(vlm_outputs)} results')

    # --- Step E: Extract Data ---
    data_for_df = []
    # ✅ This now works because row_indices is perfectly aligned with vlm_outputs
    output_lookup = {row_indices[i]: vlm_outputs[i] for i in range(len(vlm_outputs))}

    for idx, row in valid_analysis_df.iterrows():
        row_data = row.to_dict()

        if idx in missing_rows:
            for j in range(N_SAMPLES):
                row_data[f'output_sample_{j+1}'] = (
                    "FULLY_INSIDE: 0\nPARTIALLY_INSIDE: 0\nTOTAL_IMPACTED: 0\n"
                    "==================END==================\n[SKIPPED: missing images]"
                )
        elif idx in output_lookup:
            outputs = output_lookup[idx].outputs
            for j, output in enumerate(outputs):
                row_data[f'output_sample_{j+1}'] = output.text
        else:
            # ✅ Catch-all for silent vLLM failures
            for j in range(N_SAMPLES):
                row_data[f'output_sample_{j+1}'] = (
                    "FULLY_INSIDE: 0\nPARTIALLY_INSIDE: 0\nTOTAL_IMPACTED: 0\n"
                    "==================END==================\n[FAILED: vLLM inference error]"
                )

        data_for_df.append(row_data)
    # ✅ END of row loop — CSV save is now correctly outside it

    # --- Save CSV ---
    vlm_outputs_df = pd.DataFrame(data_for_df)
    safe_model_name = model_id.replace("/", "_").replace("-", "_")
    output_vlm_raw_csv = os.path.join(DRIVE_BASE, f'vlm_raw_outputs_{safe_model_name}.csv')
    vlm_outputs_df.to_csv(output_vlm_raw_csv, index=False)
    print(f"✅ Raw VLM outputs saved to: {output_vlm_raw_csv}")

    # --- Step F: Memory Cleanup ---
    print(f"[INFO] Unloading {model_id} and tearing down vLLM state...")
    del vlm_llm
    del processor
    destroy_model_parallel()

    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

    gc.collect()
    torch.cuda.empty_cache()
    print(f"[INFO] VRAM cleared.")

    # --- Step G: Disk Cleanup ---
    print(f"[INFO] Deleting model files from disk to free up storage...")
    hf_folder_name = f"models--{model_id.replace('/', '--')}"
    model_cache_path = os.path.join(os.environ["HF_HOME"], "hub", hf_folder_name)

    if os.path.exists(model_cache_path):
        shutil.rmtree(model_cache_path)
        print(f"✅ Deleted disk cache for {model_id}: {model_cache_path}")
    else:
        print(f"⚠️ Cache directory not found, skipped: {model_cache_path}")

    print(f"[INFO] Ready for next model.\n")

print("\n🎉 ALL MODELS PROCESSED AND CLEARED SUCCESSFULLY!")

✅ Drive mounted. Project folder: /content/drive/MyDrive/new/Outputs_2_no_labels
   N samples  : 5
   Temperature: 0.3
   Max tokens : 2000
✅ Environment ready (vLLM performance mode)

🚀 STARTING RUN FOR MODEL: Qwen/Qwen3.6-35B-A3B

[INFO] Downloading Qwen/Qwen3.6-35B-A3B to Disk...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

[INFO] Download complete! Model is now cached on disk.
[INFO] Loading processor: Qwen/Qwen3.6-35B-A3B
[INFO] Loading vLLM engine: Qwen/Qwen3.6-35B-A3B
INFO 06-12 04:39:00 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 2}, 'model': 'Qwen/Qwen3.6-35B-A3B'}
WARNING 06-12 04:39:00 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-12 04:39:01 [model.py:549] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 06-12 04:39:01 [model.py:1678] Using max model len 8192
INFO 06-12 04:39:01 [config.py:281] Setting attention block size to 1056 tokens to ensure that attention page size is >= mamba page size.
INFO 06-12 04:39:01 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
WARNING 06-12 04:39:01 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setti

  0%|          | 0/893 [00:00<?, ?it/s]

✅ Built 892 inputs
[INFO] Running VLM inference (n=5, temp=0.3, max_tokens=2000)...


Rendering prompts:   0%|          | 0/892 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4460 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

INFO 06-12 04:40:53 [loggers.py:259] Engine 000: Avg prompt throughput: 4544.1 tokens/s, Avg generation throughput: 1658.4 tokens/s, Running: 107 reqs, Waiting: 4352 reqs, GPU KV cache usage: 98.7%, Prefix cache hit rate: 0.0%, MM cache hit rate: 12.7%
INFO 06-12 04:41:03 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 2399.6 tokens/s, Running: 108 reqs, Waiting: 4347 reqs, GPU KV cache usage: 99.6%, Prefix cache hit rate: 0.0%, MM cache hit rate: 12.7%
INFO 06-12 04:41:13 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 2456.8 tokens/s, Running: 108 reqs, Waiting: 4346 reqs, GPU KV cache usage: 99.6%, Prefix cache hit rate: 0.0%, MM cache hit rate: 12.7%
INFO 06-12 04:41:23 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 2392.4 tokens/s, Running: 108 reqs, Waiting: 4344 reqs, GPU KV cache usage: 99.6%, Prefix cache hit rate: 0.0%, MM cache hit rate: 12.

In [ ]:
from google.colab import runtime
runtime.unassign()


## Nu Extract

In [ ]:
# ==============================================================================
# NuExtract-2.0-8B EXTRACTION PIPELINE
# ==============================================================================

import os
import gc
import re
import json
import torch
import shutil
import statistics
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
from huggingface_hub import snapshot_download


# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

os.environ["MINERU_BACKEND"] = "vllm"
os.environ["MINERU_VLLM_ENABLE"] = "true"

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"


# ==============================================================================
# SECTION 1 — CONFIG
# ==============================================================================
NUEXTRACT_MODEL_ID = "numind/NuExtract-2.0-8B"
DRIVE_BASE         = "/content/drive/MyDrive/new/Outputs_2_no_labels"
HF_HOME            = os.environ.get("HF_HOME", "/content/hf_cache")
N_SAMPLES          = 5

NUEXTRACT_TEMPLATE = json.dumps(
    {
        "Buildings_Fully_Inside":     0,
        "Buildings_Partially_Inside": 0,
        "Total_Buildings_Impacted":   0,
    },
    indent=2,
    ensure_ascii=False,
)

print(f"   Model     : {NUEXTRACT_MODEL_ID}")
print(f"   DRIVE_BASE: {DRIVE_BASE}")
print(f"   N_SAMPLES : {N_SAMPLES}")
print(f"   Template  :\n{NUEXTRACT_TEMPLATE}")


# ==============================================================================
# SECTION 2 — HELPERS
# ==============================================================================
def build_nuextract_prompt(text: str) -> str:
    return (
        "<|input|>\n"
        f"{text.strip()}\n"
        "<|schema|>\n"
        f"{NUEXTRACT_TEMPLATE}\n"
        "<|output|>\n"
    )


def safe_int(val, default: int = 0) -> int:
    try:
        return int(str(val).strip())
    except (ValueError, TypeError):
        return default


def parse_nuextract_output(raw: str) -> dict:
    for stop in ["</s>", "<|endoftext|>", "<|end|>"]:
        raw = raw.split(stop)[0]
    raw = raw.strip()

    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        try:
            obj = json.loads(m.group()) if m else {}
        except json.JSONDecodeError:
            obj = {}

    return {
        "Buildings_Fully_Inside":     safe_int(obj.get("Buildings_Fully_Inside",     0)),
        "Buildings_Partially_Inside": safe_int(obj.get("Buildings_Partially_Inside", 0)),
        "Total_Buildings_Impacted":   safe_int(obj.get("Total_Buildings_Impacted",   0)),
    }


def median_int(values: list) -> int:
    return int(round(statistics.median(values))) if values else 0


# ==============================================================================
# SECTION 3 — LOAD NuExtract WITH vLLM
# ==============================================================================
print(f"\n[INFO] Downloading {NUEXTRACT_MODEL_ID}...")
snapshot_download(repo_id=NUEXTRACT_MODEL_ID, resume_download=True, max_workers=8)
print("[INFO] Download complete.")

print(f"[INFO] Loading vLLM engine...")
nu_llm = LLM(
    model=NUEXTRACT_MODEL_ID,
    max_model_len=16384*2,
    trust_remote_code=True,
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    disable_log_stats=True,
)
print("✅ NuExtract vLLM engine ready")

nu_sampling = SamplingParams(
    temperature=0.0,
    max_tokens=256,
    stop=["</s>", "<|endoftext|>", "<|end|>"],
)


# ==============================================================================
# SECTION 4 — PROCESS CSVs
# ==============================================================================
raw_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("vlm_raw_outputs_") and f.endswith(".csv")
)
print(f"\n[INFO] Found {len(raw_csvs)} raw CSV(s): {raw_csvs}")

for csv_file in raw_csvs:
    csv_path = os.path.join(DRIVE_BASE, csv_file)
    df       = pd.read_csv(csv_path)

    print(f"\n{'='*60}")
    print(f"📄 Extracting: {csv_file}  ({len(df)} rows)")
    print(f"{'='*60}")

    sample_cols = sorted(
        [c for c in df.columns if re.match(r"output_sample_\d+$", c)],
        key=lambda c: int(c.split("_")[-1]),
    )

    prompts, row_sample_map = [], []

    for df_idx, row in df.iterrows():
        for col in sample_cols:
            text = str(row.get(col, "")).strip()
            if not text or text.lower() == "nan":
                text = "No output available."
            prompts.append(build_nuextract_prompt(text))
            row_sample_map.append((df_idx, col))

    print(f"   Total extraction prompts: {len(prompts)}")

    print("[INFO] Running NuExtract inference...")
    nu_outputs = nu_llm.generate(prompts, nu_sampling, use_tqdm=True)
    print(f"✅ Extraction done — {len(nu_outputs)} outputs")

    per_row = {}
    for (df_idx, col), out in zip(row_sample_map, nu_outputs):
        raw_text = out.outputs[0].text if out.outputs else ""
        parsed   = parse_nuextract_output(raw_text)
        per_row.setdefault(df_idx, []).append(parsed)

    consensus_rows = []

    for df_idx, row in df.iterrows():
        samples = per_row.get(df_idx, [])
        base    = row.to_dict()

        if samples:
            fully_vals   = [s["Buildings_Fully_Inside"]     for s in samples]
            partial_vals = [s["Buildings_Partially_Inside"] for s in samples]
            total_vals   = [s["Total_Buildings_Impacted"]   for s in samples]

            fully   = median_int(fully_vals)
            partial = median_int(partial_vals)
            total   = median_int(total_vals)

            fully_std   = round(statistics.pstdev(fully_vals),   4)
            partial_std = round(statistics.pstdev(partial_vals), 4)
            total_std   = round(statistics.pstdev(total_vals),   4)

            fully_range   = max(fully_vals)   - min(fully_vals)
            partial_range = max(partial_vals) - min(partial_vals)
            total_range   = max(total_vals)   - min(total_vals)

            flat_samples = {}
            for k, s in enumerate(samples):
                flat_samples[f"s{k+1}_fully"]   = s["Buildings_Fully_Inside"]
                flat_samples[f"s{k+1}_partial"] = s["Buildings_Partially_Inside"]
                flat_samples[f"s{k+1}_total"]   = s["Total_Buildings_Impacted"]

        else:
            fully = partial = total = 0
            fully_std = partial_std = total_std = 0.0
            fully_range = partial_range = total_range = 0
            flat_samples = {}

        base.update({
            "extracted_fully_inside":     fully,
            "extracted_partially_inside": partial,
            "extracted_total_impacted":   total,

            "std_fully_inside":           fully_std,
            "std_partially_inside":       partial_std,
            "std_total_impacted":         total_std,
            "range_fully_inside":         fully_range,
            "range_partially_inside":     partial_range,
            "range_total_impacted":       total_range,

            **flat_samples,

            **{
                f"nuextract_sample_{k+1}": json.dumps(s, ensure_ascii=False)
                for k, s in enumerate(samples)
            },
        })

        consensus_rows.append(base)

    out_name = csv_file.replace("vlm_raw_outputs_", "nuextract_parsed_")
    out_path = os.path.join(DRIVE_BASE, out_name)
    pd.DataFrame(consensus_rows).to_csv(out_path, index=False)
    print(f"✅ Saved → {out_path}")


# ==============================================================================
# SECTION 5 — CLEANUP
# ==============================================================================
print("\n[INFO] Unloading NuExtract...")
del nu_llm
destroy_model_parallel()

if torch.distributed.is_initialized():
    torch.distributed.destroy_process_group()

gc.collect()
torch.cuda.empty_cache()

hf_folder  = f"models--{NUEXTRACT_MODEL_ID.replace('/', '--')}"
cache_path = os.path.join(HF_HOME, "hub", hf_folder)

if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"✅ Disk cache deleted: {cache_path}")
else:
    print(f"⚠️  Cache not found, skipped: {cache_path}")

print("\n🎉 NuExtract extraction complete for all CSVs!")

   Model     : numind/NuExtract-2.0-8B
   DRIVE_BASE: /content/drive/MyDrive/new/Outputs_2_no_labels
   N_SAMPLES : 5
   Template  :
{
  "Buildings_Fully_Inside": 0,
  "Buildings_Partially_Inside": 0,
  "Total_Buildings_Impacted": 0
}

[INFO] Downloading numind/NuExtract-2.0-8B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

[INFO] Download complete.
[INFO] Loading vLLM engine...
INFO 06-12 05:49:44 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'numind/NuExtract-2.0-8B'}
WARNING 06-12 05:49:44 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-12 05:49:44 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 06-12 05:49:44 [model.py:1678] Using max model len 32768
WARNING 06-12 05:49:44 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-12 05:49:44 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-12 05:49:44 [vllm.py:1025] Cudagraph is disabled under eager mode
✅ NuExtract vLLM engine ready

[INFO] Found 3 raw CSV(s)

Rendering prompts:   0%|          | 0/4465 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/4465 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

✅ Extraction done — 4465 outputs
✅ Saved → /content/drive/MyDrive/new/Outputs_2_no_labels/nuextract_parsed_google_gemma_4_31B_it.csv

📄 Extracting: vlm_raw_outputs_sakamakismile_Huihui_Qwen3.6_35B_A3B_Claude_4.7_Opus_abliterated_NVFP4.csv  (893 rows)
   Total extraction prompts: 4465
[INFO] Running NuExtract inference...


Rendering prompts:   0%|          | 0/4465 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4465 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

✅ Extraction done — 4465 outputs
✅ Saved → /content/drive/MyDrive/new/Outputs_2_no_labels/nuextract_parsed_sakamakismile_Huihui_Qwen3.6_35B_A3B_Claude_4.7_Opus_abliterated_NVFP4.csv

[INFO] Unloading NuExtract...
✅ Disk cache deleted: /content/hf_cache/hub/models--numind--NuExtract-2.0-8B

🎉 NuExtract extraction complete for all CSVs!


In [ ]:
for csv_file in raw_csvs:
    csv_path = os.path.join(DRIVE_BASE, csv_file)
    if 'vlm_raw_outputs_Qwen_Qwen3.6_35B_A3B' not in csv_file:
      continue
    df       = pd.read_csv(csv_path)
    print(f"\n{'='*60}")
    print(f"📄 Extracting: {csv_file}  ({len(df)} rows)")
    print(f"{'='*60}")

    sample_cols = sorted(
        [c for c in df.columns if re.match(r"output_sample_\d+$", c)],
        key=lambda c: int(c.split("_")[-1]),
    )
    print(f"   Sample columns: {sample_cols}")

    # ── Flatten all (row × sample) into one batch ─────────────────────────────
    prompts        = []
    row_sample_map = []  # parallel: (df_idx, sample_col)

    for df_idx, row in df.iterrows():
        for col in sample_cols:
            text = str(row.get(col, "")).strip()
            if not text or text.lower() == "nan":
                text = "No output available."
            prompts.append(build_nuextract_prompt(text))
            row_sample_map.append((df_idx, col))

    print(f"   Total extraction prompts: {len(prompts)}")

    # ── Batched inference ─────────────────────────────────────────────────────
    print("[INFO] Running NuExtract inference...")
    nu_outputs = nu_llm.generate(prompts, nu_sampling, use_tqdm=True)
    print(f"✅ Extraction done — {len(nu_outputs)} results")

    # ── Group parsed results back by row ──────────────────────────────────────
    per_row: dict[int, list[dict]] = {}
    for (df_idx, col), out in zip(row_sample_map, nu_outputs):
        raw_text = out.outputs[0].text if out.outputs else ""
        parsed   = parse_nuextract_output(raw_text)
        per_row.setdefault(df_idx, []).append(parsed)

    # ── Build output rows ─────────────────────────────────────────────────────
    consensus_rows = []

    for df_idx, row in df.iterrows():
        samples = per_row.get(df_idx, [])
        base    = row.to_dict()

        if samples:
            fully_vals   = [s["Buildings_Fully_Inside"]     for s in samples]
            partial_vals = [s["Buildings_Partially_Inside"] for s in samples]
            total_vals   = [s["Total_Buildings_Impacted"]   for s in samples]
            map_vals     = [s["Extracted_Map_Text"]         for s in samples]

            # ── Consensus ─────────────────────────────────────────────────────
            fully    = median_int(fully_vals)
            partial  = median_int(partial_vals)
            total    = median_int(total_vals)
            map_text = majority_string(map_vals)

            # ── Spread / model uncertainty ─────────────────────────────────────
            fully_std   = round(statistics.pstdev(fully_vals),   4)
            partial_std = round(statistics.pstdev(partial_vals), 4)
            total_std   = round(statistics.pstdev(total_vals),   4)

            # ── Range (max − min) ──────────────────────────────────────────────
            fully_range   = max(fully_vals)   - min(fully_vals)
            partial_range = max(partial_vals) - min(partial_vals)
            total_range   = max(total_vals)   - min(total_vals)

            # ── Flat per-sample numerics (no JSON parsing needed later) ────────
            flat_samples = {}
            for k, s in enumerate(samples):
                flat_samples[f"s{k+1}_fully"]   = s["Buildings_Fully_Inside"]
                flat_samples[f"s{k+1}_partial"] = s["Buildings_Partially_Inside"]
                flat_samples[f"s{k+1}_total"]   = s["Total_Buildings_Impacted"]

        else:
            fully = partial = total = 0
            map_text = ""
            fully_std = partial_std = total_std = 0.0
            fully_range = partial_range = total_range = 0
            flat_samples = {}

        base.update({
            # ── Consensus ─────────────────────────────────────────────────────
            "extracted_fully_inside":     fully,
            "extracted_partially_inside": partial,
            "extracted_total_impacted":   total,
            "extracted_map_text":         map_text,

            # ── Spread metrics ─────────────────────────────────────────────────
            "std_fully_inside":           fully_std,
            "std_partially_inside":       partial_std,
            "std_total_impacted":         total_std,
            "range_fully_inside":         fully_range,
            "range_partially_inside":     partial_range,
            "range_total_impacted":       total_range,

            # ── Flat per-sample numerics ───────────────────────────────────────
            **flat_samples,

            # ── Raw JSON audit trail ───────────────────────────────────────────
            **{
                f"nuextract_sample_{k+1}": json.dumps(s, ensure_ascii=False)
                for k, s in enumerate(samples)
            },
        })
        consensus_rows.append(base)

    # ── Save parsed CSV ───────────────────────────────────────────────────────
    out_name = csv_file.replace("vlm_raw_outputs_", "nuextract_parsed_")
    out_path = os.path.join(DRIVE_BASE, out_name)
    pd.DataFrame(consensus_rows).to_csv(out_path, index=False)
    print(f"✅ Saved → {out_path}")

# ==============================================================================
# SECTION 5 — CLEANUP
# ==============================================================================
print("\n[INFO] Unloading NuExtract...")
del nu_llm
destroy_model_parallel()
if torch.distributed.is_initialized():
    torch.distributed.destroy_process_group()
gc.collect()
torch.cuda.empty_cache()

hf_folder  = f"models--{NUEXTRACT_MODEL_ID.replace('/', '--')}"
cache_path = os.path.join(HF_HOME, "hub", hf_folder)
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"✅ Disk cache deleted: {cache_path}")
else:
    print(f"⚠️  Cache not found, skipped: {cache_path}")

print("\n🎉 NuExtract extraction complete for all CSVs!")

# Eval

In [ ]:
# ==============================================================================
# SECTION — EVALUATION (INDEX-BASED, WITH STD FIX)
# ==============================================================================

import os
import re
import numpy as np
import pandas as pd

DRIVE_BASE = "/content/drive/MyDrive/Outputs_2_no_labels"

GROUND_TRUTH_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"


# ==============================================================================
# LOAD GROUND TRUTH
# ==============================================================================
print("\n📥 Loading ground truth...")
gt_df = pd.read_csv(GROUND_TRUTH_CSV)
gt_df = gt_df.reset_index(drop=True)

gt_df["gt_fully"]   = pd.to_numeric(gt_df["complete_building_count"], errors="coerce")
gt_df["gt_partial"] = pd.to_numeric(gt_df["partial_building_count"], errors="coerce")
gt_df["gt_total"]   = pd.to_numeric(gt_df["total_count"], errors="coerce")

print(f"✅ Ground truth loaded: {len(gt_df)} rows")


# ==============================================================================
# FIND PARSED FILES
# ==============================================================================
parsed_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("nuextract_parsed_") and f.endswith(".csv")
)

print(f"\n📂 Found {len(parsed_csvs)} parsed CSV(s)")

all_results = []


# ==============================================================================
# LOOP OVER MODELS
# ==============================================================================
for csv_file in parsed_csvs:

    print(f"\n📄 Processing: {csv_file}")

    df = pd.read_csv(os.path.join(DRIVE_BASE, csv_file))
    df = df.reset_index(drop=True)

    # ==============================================================================
    # SAFETY CHECK
    # ==============================================================================
    if len(df) != len(gt_df):
        print(f"   ⚠️ Row mismatch: parsed={len(df)} | gt={len(gt_df)} → skipping")
        continue

    # ==============================================================================
    # ALIGN BY INDEX
    # ==============================================================================
    merged = df.copy()
    merged["gt_fully"]   = gt_df["gt_fully"]
    merged["gt_partial"] = gt_df["gt_partial"]
    merged["gt_total"]   = gt_df["gt_total"]

    model_name = df["model_id"].iloc[0] if "model_id" in df.columns else csv_file
    temp = df["temperature"].iloc[0] if "temperature" in df.columns else 0.0

    # ==============================================================================
    # METRICS FUNCTION
    # ==============================================================================
    def metrics(pred, gt):
        diff = merged[pred] - merged[gt]
        mae  = diff.abs().mean()
        mse  = (diff ** 2).mean()
        rmse = np.sqrt(mse)
        bias = diff.mean()
        return mae, mse, rmse, bias

    mae_f, mse_f, rmse_f, bias_f = metrics("extracted_fully_inside", "gt_fully")
    mae_p, mse_p, rmse_p, bias_p = metrics("extracted_partially_inside", "gt_partial")
    mae_t, mse_t, rmse_t, bias_t = metrics("extracted_total_impacted", "gt_total")

    # ==============================================================================
    # UNCERTAINTY (STD) — FIXED ADDITION
    # ==============================================================================
    def safe_std(col):
        return merged[col].mean() if col in merged.columns else None

    std_f = safe_std("std_fully_inside")
    std_p = safe_std("std_partially_inside")
    std_t = safe_std("std_total_impacted")

    # ==============================================================================
    # TOKEN STATS
    # ==============================================================================
    tok_cols = [c for c in merged.columns if re.match(r"output_token_count_\d+$", c)]

    if tok_cols:
        toks = merged[tok_cols].values.flatten()
        mean_tokens   = float(np.mean(toks))
        median_tokens = float(np.median(toks))
        std_tokens    = float(np.std(toks))
    else:
        mean_tokens = median_tokens = std_tokens = None

    # ==============================================================================
    # SAVE RESULT
    # ==============================================================================
    all_results.append({
        "source_file": csv_file,
        "model_id": model_name,
        "temperature": temp,
        "n_events": len(merged),

        # ── TOTAL ─────────────────────────────────────────────
        "mae_total": mae_t,
        "mse_total": mse_t,
        "rmse_total": rmse_t,
        "bias_total": bias_t,
        "std_total": std_t,

        # ── FULL ──────────────────────────────────────────────
        "mae_fully": mae_f,
        "mse_fully": mse_f,
        "rmse_fully": rmse_f,
        "bias_fully": bias_f,
        "std_fully": std_f,

        # ── PARTIAL ───────────────────────────────────────────
        "mae_partial": mae_p,
        "mse_partial": mse_p,
        "rmse_partial": rmse_p,
        "bias_partial": bias_p,
        "std_partial": std_p,

        # ── TOKEN STATS ───────────────────────────────────────
        "mean_token_count": mean_tokens,
        "median_token_count": median_tokens,
        "std_token_count": std_tokens,
    })


# ==============================================================================
# FINAL OUTPUT
# ==============================================================================
eval_df = pd.DataFrame(all_results).sort_values("rmse_total")

save_path = os.path.join(DRIVE_BASE, "evaluation_summary.csv")
eval_df.to_csv(save_path, index=False)

print("\n✅ Evaluation complete!")
print(f"📁 Saved → {save_path}")

print("\n🏆 TOP MODELS:")
print(eval_df.head(10).to_string(index=False))